# 12주차 — 서브쿼리
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. 스칼라 서브쿼리

In [ ]:
# 평균보다 오래 입원한 환자
q("""
SELECT visit_id, patient_id,
       julianday(discharge_date) - julianday(visit_date) AS los
FROM visit
WHERE visit_type = '입원'
  AND julianday(discharge_date) - julianday(visit_date) >
      (SELECT AVG(julianday(discharge_date) - julianday(visit_date))
       FROM visit WHERE visit_type = '입원')
ORDER BY los DESC
""")

## 2. IN 서브쿼리

In [ ]:
# I63 진단 이력자의 인적 정보
q("""
SELECT patient_id, sex, birth_date FROM patient
WHERE patient_id IN (
  SELECT v.patient_id FROM visit v
  JOIN diagnosis d ON v.visit_id = d.visit_id
  WHERE d.kcd_code LIKE 'I63%')
""")

## 3. EXISTS / NOT EXISTS

In [ ]:
# 항생제를 한 번도 처방받지 않은 환자 수
q("""
SELECT COUNT(*) AS 무항생제환자수
FROM patient p
WHERE NOT EXISTS (
  SELECT 1 FROM visit v
  JOIN prescription rx ON rx.visit_id = v.visit_id
  WHERE v.patient_id = p.patient_id
    AND rx.drug_code IN ('CTX2','AMC625','LVX500'))
""")

In [ ]:
# 항생제 노출군과 약물
q("""
SELECT DISTINCT v.patient_id, rx.drug_name
FROM visit v JOIN prescription rx ON rx.visit_id = v.visit_id
WHERE rx.drug_code IN ('CTX2','AMC625','LVX500')
ORDER BY v.patient_id
""")

## 연습문제

In [ ]:
# [연습 Q1] 평균 투약일수보다 길게 처방된 약물 행은?
# 힌트: 스칼라 서브쿼리
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 폐렴(J18) 진단 이력이 있는 환자의 성별·생년월일은?
# 힌트: IN
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 검사(lab)를 한 번도 받지 않은 환자 수는?
# 힌트: NOT EXISTS
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 자기 진료과 평균보다 재원이 길었던 입원 건은?
# 힌트: 상관 서브쿼리 (도전)
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q5] Q3를 NOT IN으로도 써 보고, NULL이 끼면 어떻게 되는지 설명해 보라
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 평균 초과 투약**
```sql
SELECT * FROM prescription
WHERE duration_days > (SELECT AVG(duration_days) FROM prescription);
```

**A2. 폐렴 이력자**
```sql
SELECT patient_id, sex, birth_date FROM patient
WHERE patient_id IN (SELECT v.patient_id FROM visit v
  JOIN diagnosis d ON d.visit_id = v.visit_id WHERE d.kcd_code LIKE 'J18%');
```

**A3. 검사 무이력**
```sql
SELECT COUNT(*) FROM patient p WHERE NOT EXISTS (
  SELECT 1 FROM visit v JOIN lab l ON l.visit_id = v.visit_id
  WHERE v.patient_id = p.patient_id);
```

**A4. 진료과 평균 초과 재원**
```sql
SELECT v.visit_id, v.department,
  julianday(v.discharge_date)-julianday(v.visit_date) AS los
FROM visit v WHERE v.visit_type='입원'
  AND los > (SELECT AVG(julianday(v2.discharge_date)-julianday(v2.visit_date))
             FROM visit v2 WHERE v2.visit_type='입원' AND v2.department = v.department);
```